This is a kaggle competition (March, 2026) dataset taken from kaggle.
Using it to validate the trained XGboost model and selected threshold value 0.33.

In [1]:
import numpy as np
import pandas as pd

In [2]:
train_df = pd.read_csv("Dataset/kaggle_competition_dataset//train.csv")

In [3]:
test_df = pd.read_csv("Dataset/kaggle_competition_dataset//test.csv")

In [4]:
train_df.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 
 16  PaperlessBilling  59

In [6]:
summary_rows = []

n_rows = len(train_df)

for column_name, column_data in train_df.items():

    missing_count = column_data.isna().sum()
    missing_ratio = (missing_count / n_rows) * 100

    example_values = column_data.dropna().unique()[:3]
    example_preview = ", ".join(map(str, example_values))[:60]

    summary_rows.append({
        "column_name": column_name,
        "dtype": column_data.dtype,
        "unique_values": column_data.nunique(),
        "missing_count": missing_count,
        "missing_percent": round(missing_ratio, 2),
        "example_values": example_preview
    })

dataset_summary = pd.DataFrame(summary_rows)

display(
    dataset_summary.style.set_properties(**{"text-align": "left"})
)

,column_name,dtype,unique_values,missing_count,missing_percent,example_values
0,id,int64,594194,0,0.000000,"0, 1, 2"
1,gender,object,2,0,0.000000,"Male, Female"
2,SeniorCitizen,int64,2,0,0.000000,"0, 1"
3,Partner,object,2,0,0.000000,"Yes, No"
4,Dependents,object,2,0,0.000000,"Yes, No"
5,tenure,int64,72,0,0.000000,"29, 58, 1"
6,PhoneService,object,2,0,0.000000,"Yes, No"
7,MultipleLines,object,3,0,0.000000,"No, Yes, No phone service"
8,InternetService,object,3,0,0.000000,"DSL, Fiber optic, No"
9,OnlineSecurity,object,3,0,0.000000,"Yes, No, No internet service"


In [7]:
train_df.duplicated().sum()

0

In [8]:
train_df["id"].duplicated().sum()

0

In [9]:
train_df['SeniorCitizen'] = train_df['SeniorCitizen'].map({
    0: 'No',
    1: 'Yes'
})

In [10]:
train_df.dropna(inplace = True)

In [11]:
labels = ["{0} - {1}".format(i, i + 11) for i in range(1, 72, 12)]
train_df['tenure_group'] = pd.cut(train_df.tenure, range(1, 80, 12), right=False, labels=labels)

train_df['tenure_group'].value_counts()

tenure_group
61 - 72    158107
1 - 12     152646
13 - 24     86024
49 - 60     69429
25 - 36     66175
37 - 48     61813
Name: count, dtype: int64

In [12]:
train_df["Churn"].value_counts()

Churn
No     460377
Yes    133817
Name: count, dtype: int64

In [13]:
df = train_df.drop(columns=['tenure'])

In [14]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

In [15]:
y = y.map({
    "No": 0,
    "Yes": 1
})

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [17]:
import joblib
preprocessor = joblib.load("models/preprocessor.pkl")

In [18]:
X_processed = preprocessor.transform(X)

In [19]:
model = joblib.load("models/tuned_xgboost.pkl")

In [20]:
X_train = joblib.load("Dataset/X_train.pkl")
X_test = joblib.load("Dataset/X_test.pkl")

y_train = joblib.load("Dataset/y_train.pkl")
y_test = joblib.load("Dataset/y_test.pkl")

In [21]:
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.4, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=2, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=-1,
              num_parallel_tree=None, ...)

In [22]:
y_prob = model.predict_proba(X_processed)[:, 1]

In [23]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [24]:
y_pred_default = (
    y_prob >= 0.50
).astype(int)

default_results = {
    "Threshold": 0.50,
    "Accuracy": accuracy_score(
        y,
        y_pred_default
    ),
    "Precision": precision_score(
        y,
        y_pred_default
    ),
    "Recall": recall_score(
        y,
        y_pred_default
    ),
    "F1": f1_score(
        y,
        y_pred_default
    ),
    "ROC-AUC" : roc_auc_score(
        y,
        y_pred_default
    )
}

y_pred = (
    y_prob >= 0.30
).astype(int)

optimized_results = {
    "Threshold": 0.33,
    "Accuracy": accuracy_score(
        y,
        y_pred
    ),
    "Precision": precision_score(
        y,
        y_pred
    ),
    "Recall": recall_score(
        y,
        y_pred
    ),
    "F1": f1_score(
        y,
        y_pred
    ),
    "ROC-AUC" : roc_auc_score(
        y,
        y_pred
    )
}

comparison = pd.DataFrame([
    default_results,
    optimized_results
])

comparison

,Threshold,Accuracy,Precision,Recall,F1,ROC-AUC
0,0.50,0.849566,0.650727,0.716710,0.682126,0.802447
1,0.33,0.797440,0.529789,0.894236,0.665377,0.831770


### Optimized Classification Report

In [25]:
from sklearn.metrics import classification_report
print(
    classification_report(
        y,
        y_pred,
        target_names=["No Churn", "Churn"]
    )
)

              precision    recall  f1-score   support

    No Churn       0.96      0.77      0.85    460377
       Churn       0.53      0.89      0.67    133817

    accuracy                           0.80    594194
   macro avg       0.75      0.83      0.76    594194
weighted avg       0.86      0.80      0.81    594194



### Default Classification Report

In [26]:
from sklearn.metrics import classification_report
print(
    classification_report(
        y,
        y_pred_default,
        target_names=["No Churn", "Churn"]
    )
)

              precision    recall  f1-score   support

    No Churn       0.92      0.89      0.90    460377
       Churn       0.65      0.72      0.68    133817

    accuracy                           0.85    594194
   macro avg       0.78      0.80      0.79    594194
weighted avg       0.86      0.85      0.85    594194



### Optimized Confusion Matrix

In [27]:
from sklearn.metrics import confusion_matrix
print(
    confusion_matrix(
        y,
        y_pred
    )
)

[[354170 106207]
 [ 14153 119664]]


## Default Confusion Matrix

In [28]:
from sklearn.metrics import confusion_matrix
print(
    confusion_matrix(
        y,
        y_pred_default
    )
)

[[408899  51478]
 [ 37909  95908]]
